In [1]:
import json
import sys
from collections import Counter
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from pydantic import ValidationError
from src.schemas import TicketRoutingOutput

def read_jsonl(path: Path) -> list[dict]:
    rows = []

    with path.open("r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))

    return rows

PRED_PATH = PROJECT_ROOT / "predictions" / "prompt_few_shot_qwen.jsonl"
GOLD_PATH = PROJECT_ROOT / "data" / "processed" / "test.jsonl"

predictions  = read_jsonl(PRED_PATH)
gold_rows = read_jsonl(GOLD_PATH)

print(f"Predictions: {len(predictions )}")
print(f"Gold rows: {len(gold_rows)}")

assert len(predictions) == len(gold_rows)

Predictions: 150
Gold rows: 150


In [2]:
topic_counter = Counter()

for pred in predictions:
    parsed = pred["parsed_response"]

    if not isinstance(parsed, dict):
        continue

    topic = parsed.get("topic", "__MISSING__")
    topic_counter[topic] += 1

print("Most common predicted topics: \n")

for topic, count in topic_counter.most_common():
    print(f"{count:>3} | {topic}")

Most common predicted topics: 

106 | General Inquiry
 15 | IT Support
  9 | Technical Support
  6 | Billing and Payments
  3 | Customer Service
  2 | Returns and Exchanges
  2 | Human Resources
  1 | Marketing
  1 | Product Support
  1 | Digital Branding
  1 | Healthcare
  1 | Documentation
  1 | Pre-Sales Query
  1 | Sales and Pre-Sales


In [3]:
general_inquiry_count = topic_counter["General Inquiry"]
general_inquiry_share = general_inquiry_count / len(predictions)

print(f"General Inquiry: {general_inquiry_count}/{len(predictions)}")
print(f"Share: {general_inquiry_share:.2%}")

General Inquiry: 106/150
Share: 70.67%


In [4]:
invalid_rows = [
    row
    for row in predictions
    if row["schema_valid"] == False
]

print(f"Schema-invalid rows: {len(invalid_rows)}")

error_counter = Counter()

for row in invalid_rows:
    parsed = row["parsed_response"]

    if not isinstance(parsed, dict):
        error_counter["Response is not a JSON object"] += 1
        continue

    try:
        TicketRoutingOutput.model_validate(parsed)
    except ValidationError as exc:
        for error in exc.errors():
            field = '.'.join(str(part) for part in error["loc"])
            message = error["msg"]

            error_counter[f"{field}: {message}"] += 1

print("\nSchema violation reasons:\n")

for error, count in error_counter.most_common():
    print(f"{count:>3} | {error}")

Schema-invalid rows: 14

Schema violation reasons:

  5 | topic: Input should be 'Billing and Payments', 'Customer Service', 'General Inquiry', 'Human Resources', 'IT Support', 'Product Support', 'Returns and Exchanges', 'Sales and Pre-Sales', 'Service Outages and Maintenance' or 'Technical Support'
  4 | ticket_type: Input should be 'Incident', 'Request', 'Problem' or 'Change'
  4 | tags: List should have at most 8 items after validation, not 9
  1 | tags: List should have at most 8 items after validation, not 11


In [5]:
tag_counts = []
missing_tags_count = 0
non_list_tags_count = 0

for row in predictions:
    parsed = row["parsed_response"]

    if not isinstance(parsed, dict):
        continue

    if "tags" not in parsed:
        missing_tags_count += 1
        continue

    tags = parsed["tags"]

    if isinstance(tags, list):
        tag_counts.append(len(tags))
    else:
        non_list_tags_count += 1

average_tag_count = sum(tag_counts) / len(tag_counts)
too_many_tags_count = sum(count > 8 for count in tag_counts)

print(f"Responses with tags as list: {len(tag_counts)}")
print(f"Responses with missing tags: {missing_tags_count}")
print(f"Responses where tags is not a list: {non_list_tags_count}")
print(f"Average number of tags: {average_tag_count:.2f}")
print(f"Responses with more than 8 tags: {too_many_tags_count}")

print("\nTag count distribution:")
print(sorted(Counter(tag_counts).items()))

Responses with tags as list: 150
Responses with missing tags: 0
Responses where tags is not a list: 0
Average number of tags: 4.63
Responses with more than 8 tags: 5

Tag count distribution:
[(0, 9), (2, 3), (3, 22), (4, 34), (5, 37), (6, 28), (7, 10), (8, 2), (9, 4), (11, 1)]


In [6]:
ticket_type_confusions = Counter()
topic_confusions = Counter()

for gold_row, pred_row in zip(gold_rows, predictions):
    gold = gold_row["target"]
    parsed = pred_row["parsed_response"]

    if isinstance(parsed, dict):
        predicted_ticket_type = parsed.get("ticket_type", "__MISSING__")
        predicted_topic = parsed.get("topic", "__MISSING__")
    else:
        predicted_ticket_type = "__NOT_JSON_OBJECT__"
        predicted_topic = "__NOT_JSON_OBJECT__"

    if predicted_ticket_type != gold["ticket_type"]:
        ticket_type_confusions[
            (gold["ticket_type"], predicted_ticket_type)
        ] += 1

    if predicted_topic != gold["topic"]:
        topic_confusions[
            (gold["topic"], predicted_topic)
        ] += 1

print("Most common ticket_type confusions:\n")

for (gold_value, predicted_value), count in ticket_type_confusions.most_common(20):
    print(f"{count:>3} | gold={gold_value} → predicted={predicted_value}")

print("\nMost common topic confusions:\n")

for (gold_value, predicted_value), count in topic_confusions.most_common(30):
    print(f"{count:>3} | gold={gold_value} → predicted={predicted_value}")

Most common ticket_type confusions:

 38 | gold=Incident → predicted=Problem
 10 | gold=Change → predicted=Request
  9 | gold=Incident → predicted=Request
  4 | gold=Problem → predicted=Request
  2 | gold=Problem → predicted=Incident
  2 | gold=Request → predicted=Change
  1 | gold=Problem → predicted=Change
  1 | gold=Incident → predicted=Issue Report
  1 | gold=Incident → predicted=Issue
  1 | gold=Incident → predicted=Issue with Automated Rebalancing Systems
  1 | gold=Request → predicted=Advice
  1 | gold=Change → predicted=Incident

Most common topic confusions:

 32 | gold=Technical Support → predicted=General Inquiry
 29 | gold=Product Support → predicted=General Inquiry
 14 | gold=Customer Service → predicted=General Inquiry
  8 | gold=IT Support → predicted=General Inquiry
  8 | gold=Service Outages and Maintenance → predicted=General Inquiry
  5 | gold=Billing and Payments → predicted=General Inquiry
  4 | gold=Product Support → predicted=IT Support
  4 | gold=Sales and Pre-S